In [1]:
import os
import pandas as pd

In [2]:
MODE = "PDE_TIER2" # Change this to "PDE_TIER1" or "PDE_TIER2" as needed

# PDE Operationalisation

In [3]:
def get_score_diff(events, team, t_seconds):
    
    # < to not count the current event if it's a goal
    past_events = events[events.minute * 60 + events.second < t_seconds]
    
    if past_events.empty:
        return 0
        
    # Normal goals
    if 'shot_outcome' in past_events.columns:
        normal_goals = past_events[(past_events['type'] == 'Shot') & (past_events['shot_outcome'] == 'Goal')]
    else:
        normal_goals = pd.DataFrame()
        
    # Own goals
    own_goals = past_events[past_events['type'] == 'Own Goal Against']
    
    # Goals of affected team
    team_scored = len(normal_goals[normal_goals['team'] == team])
    team_benefited = len(own_goals[own_goals['team'] != team])
    total_team_goals = team_scored + team_benefited
    
    # Goals of opposing team
    opp_scored = len(normal_goals[normal_goals['team'] != team])
    opp_benefited = len(own_goals[own_goals['team'] == team])
    total_opp_goals = opp_scored + opp_benefited
    
    # Goal difference from the perspective of the affected team
    return total_team_goals - total_opp_goals

## PDE 1 - Big Miss
Shots with high xG that did not result in a goal

In [4]:
def detect_big_misses(events, match_id, xg_threshold=0.20):
    
    # Filter: Non-penalty shots that did not result in a goal
    potential_misses = events[
        (events.type == 'Shot') & 
        (events.shot_type != 'Penalty') & 
        (events.shot_outcome != 'Goal') &
        (events.shot_statsbomb_xg >= xg_threshold) # Minimum threshold for robustness 
    ].copy()
    
    misses_list = []
    for _, shot in potential_misses.iterrows():
        misses_list.append({
            'event_id': shot.id,
            'match_id': match_id,
            'period': shot.period,
            'minute': shot.minute,
            'second': shot.second,
            'total_seconds': shot.minute * 60 + shot.second,
            'goal_diff': get_score_diff(events, shot.team, shot.minute * 60 + shot.second),
            'event_type': 'Big Miss',
            'affected_team': shot.team,
            'severity': shot.shot_statsbomb_xg, # Using xG as severity
        })
    return misses_list

## PDE 2 - Selfish Shot

In [5]:
def count_goal_blockers(origin_loc, freeze_frame):

    if not isinstance(freeze_frame, list):
        return 0
    
    x_s, y_s = origin_loc
    blockers = 0
    
    # Goal is at x=120, from y=36 to y=44
    for player in freeze_frame:
        if not player['teammate']: # Only opponents matter
            x_o, y_o = player['location']
            
            # Is the opponent between the player and the goal?
            if x_o > x_s:
                # Triangle check: Is the opponent inside the 'scoring cone'?
                
                progress = (x_o - x_s) / (120 - x_s + 1e-9)
                y_min_at_x = y_s + (36 - y_s) * progress
                y_max_at_x = y_s + (44 - y_s) * progress
                
                # Check if opponent is within the cone (with a small 0.5m buffer)
                if (min(y_min_at_x, y_max_at_x) - 0.5) <= y_o <= (max(y_min_at_x, y_max_at_x) + 0.5):
                    blockers += 1
                    
    return blockers

In [6]:
from sklearn.linear_model import LogisticRegression
import numpy as np

def train_simple_xg_model(all_shots):
    
    # Train on all shots
    shots = all_shots.copy()

    # Basic features
    shots['dist'] = shots.location.apply(lambda loc: np.sqrt((120 - loc[0])**2 + (40 - loc[1])**2))
    shots['angle'] = shots.location.apply(lambda loc: np.abs(np.arctan2(40 - loc[1], 120 - loc[0])))
    shots['blockers'] = shots.apply(lambda row: count_goal_blockers(row.location, row.shot_freeze_frame), axis=1)
    
    # Target variable: 1 for Goal, 0 for non-goal
    X = shots[['dist', 'angle', 'blockers']]
    y = (shots.shot_outcome == 'Goal').astype(int)
    
    model = LogisticRegression()
    model.fit(X, y)
    return model

In [7]:
def get_dist_to_line(p1, p2, opp):
    numerator = np.abs((p2[0]-p1[0])*(p1[1]-opp[1]) - (p1[0]-opp[0])*(p2[1]-p1[1]))
    denominator = np.linalg.norm(p2-p1)
    return numerator / (denominator + 1e-9)

In [8]:
def is_pass_feasible(shooter_pos, teammate_pos, freeze_frame):
    
    dist = np.linalg.norm(np.array(shooter_pos) - np.array(teammate_pos))
    if dist > 25:
        return False
    
    opponents = [p['location'] for p in freeze_frame if not p['teammate']]
    for opp in opponents:
        # Distance from point (opponent) to line segment (pass)
        p1, p2 = np.array(shooter_pos), np.array(teammate_pos)
        d = get_dist_to_line(p1, p2, opp)
        if d < 2.0: # 2m corridor check
            return False
    return True

In [9]:
def detect_selfish_shots(events, xg_model, match_id):

    selfish_list = []
    
    # Shooter xG <= 0.08 and Freeze Frame exists
    potential_shots = events[
        (events.type == 'Shot') & 
        (events.shot_statsbomb_xg <= 0.08) & 
        (events.shot_freeze_frame.notnull())
    ].copy()
    
    for _, shot in potential_shots.iterrows():
        shooter_pos = shot.location
        freeze_frame = shot.shot_freeze_frame
        
        teammates = [p for p in freeze_frame if p['teammate']]
        
        for tm in teammates:

            # Feature extraction for the teammate
            tm_pos = tm['location'] 
            tm_dist = np.sqrt((120 - tm_pos[0])**2 + (40 - tm_pos[1])**2)
            tm_angle = np.abs(np.arctan2(40 - tm_pos[1], 120 - tm_pos[0]))
            tm_blockers = count_goal_blockers(tm_pos, freeze_frame)
            
            # Estimate teammate xG
            features = pd.DataFrame([[tm_dist, tm_angle, tm_blockers]], 
                        columns=['dist', 'angle', 'blockers'])
            tm_xg_pred = xg_model.predict_proba(features)[0][1]
            
            # Check if there is a significant advantage in passing to this teammate
            if tm_xg_pred >= 0.15:
                if is_pass_feasible(shooter_pos, tm_pos, freeze_frame):
                    selfish_list.append({
                        'event_id': shot.id,
                        'match_id': match_id,
                        'period': shot.period,
                        'minute': shot.minute,
                        'second': shot.second,
                        'total_seconds': shot.minute * 60 + shot.second,
                        'goal_diff': get_score_diff(events, shot.team, shot.minute * 60 + shot.second),
                        'event_type': 'Selfish Shot',
                        'affected_team': shot.team,
                        'severity': tm_xg_pred - shot.shot_statsbomb_xg # Advantage gained if passed
                    })
                    break # Found at least one better option
                    
    return selfish_list

## PDE 3 - Yellow Card

In [10]:
def get_yellow_card_severity(card):
    position = str(card.get('position') or card.get('player_position') or '').lower()
    position = position.replace('/', ' ').replace('-', ' ').replace('_', ' ')
    position = ' '.join(position.split())

    defender_positions = {
        'center back', 'left back', 'left center back', 'left wing back',
        'right back', 'right center back', 'right wing back'
    }
    defensive_midfield_positions = {
        'center defensive midfield', 'left defensive midfield', 'right defensive midfield'
    }
    other_midfield_positions = {
        'center attacking midfield', 'center midfield', 'left attacking midfield',
        'left center midfield', 'left midfield', 'right attacking midfield',
        'right center midfield', 'right midfield'
    }
    forward_positions = {
        'center forward', 'striker', 'left center forward', 'left wing',
        'right center forward', 'right wing', 'secondary striker'
    }

    if position in defender_positions:
        return 0.8
    if position in defensive_midfield_positions:
        return 0.6
    if position in other_midfield_positions:
        return 0.5
    if position in forward_positions:
        return 0.2
    if position == 'goalkeeper':
        return 0.5
    return 0.5


def detect_yellow_cards(events, match_id):

    yellow_list = []
    
    # StatsBomb stores card data in these two specific columns depending on the foul type
    cols_to_check = ['foul_committed_card', 'bad_behaviour_card']
    
    for col in cols_to_check:
        if col in events.columns:
            # Already filters second yellow
            cards = events[events[col] == 'Yellow Card']
            
            for _, card in cards.iterrows():
                yellow_list.append({
                    'event_id': card.id,
                    'match_id': match_id,
                    'period': card.period,
                    'minute': card.minute,
                    'second': card.second,
                    'total_seconds': card.minute * 60 + card.second,
                    'goal_diff': get_score_diff(events, card.team, card.minute * 60 + card.second),
                    'event_type': 'Yellow Card',
                    'affected_team': card.team,
                    'severity': get_yellow_card_severity(card),
                })
                
    return yellow_list

## PDE 4 - Shock Goals

In [11]:
def detect_shock_goals(events, match_id, window_seconds=300):
    
    # Identify all goals in the match
    goals = events[(events.type == 'Shot') & (events.shot_outcome == 'Goal')].copy()
    
    shock_goals_list = []
    
    for _, goal in goals.iterrows():
        goal_time = goal.minute * 60 + goal.second # Time of the goal in seconds
        
        # Identify scoring and conceding teams
        scoring_team = goal.team
        all_teams = events.team.unique()
        conceding_team = [t for t in all_teams if t != scoring_team][0]
        
        # Define the window before the goal
        window_start = max(0, goal_time - window_seconds)
        window_events = events[
            (events.minute * 60 + events.second >= window_start) &
            (events.minute * 60 + events.second < goal_time) &
            (events.period == goal.period) # Ensure we stay within the same half
        ]
        
        if window_events.empty:
            continue
            
        # Calculate cumulative xG for both teams in the window
        window_shots = window_events[window_events.type == 'Shot']
        xg_conceding_team = window_shots[window_shots.team == conceding_team].shot_statsbomb_xg.sum()
        xg_scoring_team = window_shots[window_shots.team == scoring_team].shot_statsbomb_xg.sum()
        
        # Calculate Possession Share
        # Using event count as a proxy for possession duration per team
        events_conceding = len(window_events[window_events.possession_team == conceding_team])
        total_window_events = len(window_events)
        possession_share = events_conceding / total_window_events if total_window_events > 0 else 0
        
        # Apply Shock Goal Logic
        xg_ratio_threshold = xg_conceding_team >= (2 * xg_scoring_team)
        if possession_share >= 0.60 and xg_ratio_threshold:

            possession_diff = possession_share - (1 - possession_share)
            shock_goals_list.append({
                'event_id': goal.id,
                'match_id': match_id,
                'period': goal.period,
                'minute': goal.minute,
                'second': goal.second,
                'total_seconds': goal.minute * 60 + goal.second,
                'goal_diff': get_score_diff(events, conceding_team, goal.minute * 60 + goal.second),
                'event_type': 'Shock Goal',
                'affected_team': conceding_team, # The team that conceded despite dominating
                'severity': possession_diff, # Differential at moment of goal 
            })
            
    return shock_goals_list

## PDE 5 - Wasted Turnovers

In [12]:
def detect_wasted_turnovers(events, match_id):
    turnover_list = []

    events['total_seconds'] = events['minute'] * 60 + events['second']
    
    # Filter for ball recoveries in the attacking third (x > 80)
    recoveries = events[
        (events.type.isin(['Ball Recovery', 'Interception'])) & 
        (events.location.apply(lambda loc: loc[0] if isinstance(loc, list) else 0) > 80)
    ].copy().sort_values(['period', 'minute', 'second'])
    
    last_pde_time = {} 

    for _, recovery in recoveries.iterrows():
        recovery_time = recovery.total_seconds
        team = recovery.team
        period = recovery.period
        
        # Frequency filter: 5-minute window (300 seconds)
        if team in last_pde_time and (recovery_time - last_pde_time[team]) < 300:
            continue
        
        # Define the 15-second window after the recovery
        window_events = events[
            (events.total_seconds > recovery_time) & 
            (events.total_seconds <= recovery_time + 15) & 
            (events.period == period)
        ]
        
        # Check for any shots by the recovering team in this window
        has_shot = any(window_events[window_events.team == team].type == 'Shot')
        
        # PDE condition: NO SHOT AND LOST POSSESSION within 15 seconds
        if not has_shot:
            # Check if possession was lost (any event in the window where the possession team is not the recovering team)
            lost_possession = any(window_events.possession_team != recovery.possession_team)
            if lost_possession:
                turnover_list.append({
                    'event_id': recovery.id,
                    'match_id': match_id,
                    'period': recovery.period,
                    'minute': recovery.minute,
                    'second': recovery.second,
                    'total_seconds': recovery_time,
                    'goal_diff': get_score_diff(events, team, recovery_time),
                    'event_type': 'Wasted Turnover',
                    'affected_team': team,
                    'severity': recovery.location[0]
                })
            last_pde_time[team] = recovery_time
            
    return turnover_list

In [2]:
import glob

# Get all match IDs from the offline data folder
if MODE == "PDE_TIER1":
    files = glob.glob("sb_offline_data/match_*.pkl") # Load match files for TIER1 data
elif MODE == "PDE_TIER2":
    files = glob.glob("sb_offline_360_data/events_*.pkl") # Load match files for TIER2 data

match_ids = [int(os.path.basename(f).split('_')[1].split('.')[0]) for f in files]

print(f"All {len(match_ids)} match IDs found.")

NameError: name 'MODE' is not defined

In [14]:
# xG model training
all_shots_list = []

for m_id in match_ids:
    if MODE == "PDE_TIER1":
        events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
    elif MODE == "PDE_TIER2":
        events = pd.read_pickle(f"sb_offline_360_data/events_{m_id}.pkl")
    match_shots = events[events.type == 'Shot'].copy()
    all_shots_list.append(match_shots)

all_shots_df = pd.concat(all_shots_list, ignore_index=True)
xg_model = train_simple_xg_model(all_shots_df)

In [15]:
def filter_isolated_pdes(pde_df, window_seconds=300):

    if pde_df.empty:
        return pde_df
    
    # Sort by match and time
    df = pde_df.sort_values(['match_id', 'total_seconds']).copy()
    
    # Time difference to the previous PDE in the same match
    df['prev_diff'] = df.groupby('match_id')['total_seconds'].diff()
    
    # Time difference to the next PDE in the same match
    df['next_diff'] = df.groupby('match_id')['total_seconds'].diff(-1).abs()
    
    # Mask for isolated PDEs: both previous and next differences must be greater than the window
    is_isolated = (df['prev_diff'].fillna(9999) > window_seconds) & \
                  (df['next_diff'].fillna(9999) > window_seconds)
    
    # Keep only isolated PDEs and drop the diff columns
    filtered_df = df[is_isolated].drop(columns=['prev_diff', 'next_diff'])
    
    return filtered_df

In [16]:
def filter_full_window_pdes(isolated_df, window_seconds=300):
    
    valid_indices = []
    
    # Going through each match to check if the PDEs have a full window before and after within the same half
    for m_id, group in isolated_df.groupby('match_id'):
        if MODE == "PDE_TIER1":
            events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
        elif MODE == "PDE_TIER2":
            events = pd.read_pickle(f"sb_offline_360_data/events_{m_id}.pkl")
        
        events['total_seconds'] = events['minute'] * 60 + events['second']
        
        # Calculate the time bounds of each period (half) for this match
        period_bounds = events.groupby('period')['total_seconds'].agg(['min', 'max']).to_dict('index')
        
        for idx, row in group.iterrows():
            t = row['total_seconds']
            period = row['period']
            
            if period in period_bounds:
                p_start = period_bounds[period]['min']
                p_end = period_bounds[period]['max']
                
                # Check if there's enough time before and after
                has_full_pre = (t - p_start) >= window_seconds
                has_full_post = (p_end - t) >= window_seconds
                
                if has_full_pre and has_full_post:
                    valid_indices.append(idx)
                    
    # Only return the rows with valid indices
    final_df = isolated_df.loc[valid_indices].copy()
    
    return final_df

In [17]:
def summarize_pdes(pde_df):
    
    print(f"Total PDEs detected: {len(pde_df)}")

    print(f"Number of PDEs with severity >= 0.30: {len(pde_df[(pde_df['severity'] >= 0.30) & (pde_df['event_type'] == 'Big Miss')])}")

    print(f"Number of Selfish Shots: {len(pde_df[pde_df['event_type'] == 'Selfish Shot'])}")

    print(f"Number of Yellow Cards: {len(pde_df[pde_df['event_type'] == 'Yellow Card'])}")

    print(f"Number of Shock Goals: {len(pde_df[pde_df['event_type'] == 'Shock Goal'])}")

    print(f"Number of Wasted Turnovers: {len(pde_df[pde_df['event_type'] == 'Wasted Turnover'])}")

## PDE Pipeline

In [1]:
all_pdes = []

print(f"Processing {len(match_ids)} matches...")

for i, m_id in enumerate(match_ids):
    if MODE == "PDE_TIER1":
        events = pd.read_pickle(f"sb_offline_data/match_{m_id}.pkl")
    elif MODE == "PDE_TIER2":
        events = pd.read_pickle(f"sb_offline_360_data/events_{m_id}.pkl")
    
    # 1. Detect Big Misses (PDE-1)
    match_big_misses = detect_big_misses(events, m_id, 0.30)
    all_pdes.extend(match_big_misses)
    
    # 2. Detect Selfish Shots (PDE-2)
    match_selfish_shots = detect_selfish_shots(events, xg_model, m_id)
    all_pdes.extend(match_selfish_shots)

    # 3. Detect Yellow Cards (PDE-3)
    match_yellow_cards = detect_yellow_cards(events, m_id)
    all_pdes.extend(match_yellow_cards)

    # 4. Detect Shock Goals (PDE-4)
    match_shock_goals = detect_shock_goals(events, m_id)
    all_pdes.extend(match_shock_goals)

    # 5. Detect Wasted Turnovers (PDE-5)
    match_wasted_turnovers = detect_wasted_turnovers(events, m_id)
    all_pdes.extend(match_wasted_turnovers)

    if (i + 1) % 100 == 0:
        print(f"Progress: {i + 1}/{len(match_ids)} matches completed.")

pde_all_df = pd.DataFrame(all_pdes)
summarize_pdes(pde_all_df)

if MODE == "PDE_TIER1":
    pde_all_df.to_csv('output/pde/catalogue/events_raw.csv', index=False, encoding='utf-8-sig')
    pde_df = filter_isolated_pdes(pde_all_df)
    summarize_pdes(pde_df)
    pde_df = filter_full_window_pdes(pde_df)
    summarize_pdes(pde_df)
    pde_df.to_csv('output/pde/catalogue/events_filtered.csv', index=False, encoding='utf-8-sig')
elif MODE == "PDE_TIER2":
    pde_all_df.to_csv('output/pde360/catalogue/events_raw.csv', index=False, encoding='utf-8-sig')
    pde_df = filter_isolated_pdes(pde_all_df)
    summarize_pdes(pde_df)
    pde_df = filter_full_window_pdes(pde_df)
    summarize_pdes(pde_df)
    pde_df.to_csv('output/pde360/catalogue/events_filtered.csv', index=False, encoding='utf-8-sig')

NameError: name 'match_ids' is not defined